<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c42-header.png" alt="Nextia Learning · Vision AI: See, Count and Inspect" width="100%">

# Solution: Boxes and IoU

**[Boxes and IoU](https://learning.nextia-ai.com/courses/vision/m04/boxes-and-iou/)** · Vision AI: See, Count and Inspect · Module 4, lesson 1 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** read a detector's boxes, judge a box with intersection over union (IoU), remove double boxes with non-maximum suppression (NMS), and count boxes on a pallet.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells. No account, no key and no model download. |
| **You should know** | The five kinds of answer, from [Five kinds of answer](https://learning.nextia-ai.com/courses/vision/m01/five-kinds-of-answer/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, Pillow 13.0.0, scikit-learn 1.9.1 and matplotlib 3.11.2 (PyTorch 2.15.0 and timm 1.0.31 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/vision/m04/boxes-and-iou/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C42/M04-L01-boxes-and-iou/boxes-and-iou-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, pillow 13.0.0, scikit-learn 1.9.1, scikit-image 0.26.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 pillow==13.0.0 scikit-learn==1.9.1 scikit-image==0.26.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import numpy; print("numpy", numpy.__version__)
import pandas; print("pandas", pandas.__version__)
import PIL; print("pillow", PIL.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import skimage; print("scikit-image", skimage.__version__)

## Setup: get the Larkfield images

Run the next cell. It downloads the image sets that this lesson needs (about 6 MB, as zip files) into a folder `larkfield-images`, with the course's helper `compare.py`, checks each file's checksum and unpacks it. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-images` (in Colab: the **Files** panel on the left), then run the cell again.

Where the photos come from: the catalogue photos are cut from Open Images photos (CC BY 2.0: the author of each photo is in `data/catalogue/manifest.csv`), the returned candles are photos of the VisA inspection set (CC BY 4.0), and the pallets are drawn by our code. Read `data/dataset-card.md` for the details.

In [ ]:
# Setup: download the image sets into larkfield-images/, check them and unpack them.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C42/"
FILES = {
    "archives/pallets.zip": "cec4b211fa4e434936fc0c1980accfc74392f1ced9de6c04b5421b9ec6af56e3",
    "archives/extras.zip": "f121137bc765866c25929a4233c557ba72b64eea0c806070cc04a86c28ec27ec",
    "compare.py": "65e58d2fe043105e3ff09626861a8f64e5548039df760ea10d6fc02c71f8fec3",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-images").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-images and run again.")
    if name.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(PROJECT / "data")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
n = sum(1 for p in Path("data").rglob("*") if p.is_file())
print(f"Ready: {n} files in larkfield-images/data.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, Pillow's `Image`, matplotlib, and the course helper as `cp` (the comparison table). It defines `show(paths)`, which draws photos in a row, and the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: images, the comparison table, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import compare as cp
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
CLASSES = ["bowl", "candle", "chair", "cushion", "lamp", "mug", "plant_pot", "vase"]

def show(paths, titles=None, size=2.2):
    """Draw photos in a row (at most 8)."""
    paths = list(paths)[:8]
    fig, axes = plt.subplots(1, len(paths), figsize=(size * len(paths), size))
    for i, (ax, p) in enumerate(zip(np.atleast_1d(axes), paths)):
        ax.imshow(Image.open(p)); ax.axis("off")
        if titles:
            ax.set_title(str(titles[i]), fontsize=8)
    plt.show()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A box is four numbers

A box is `[x, y, w, h]`: the left edge, the top edge, the width and the height, in pixels. The pallet photos are drawn by our code, so we know every true box.

In [ ]:
pallets = json.load(open("data/pallets/test.json"))
images = {im["scene_id"]: im for im in pallets["images"]}
truth = {}
for a in pallets["annotations"]:
    sid = next(im["scene_id"] for im in pallets["images"] if im["id"] == a["image_id"])
    truth.setdefault(sid, []).append(a["bbox"])
preds = json.load(open("data/predictions/pallets_fasterrcnn_finetuned_40.json"))
print(len(images), "photos,", sum(len(v) for v in truth.values()), "true boxes,", len(preds), "recorded boxes")
print(preds[0])

The recorded boxes come from a Faster R-CNN detector that the course team fine-tuned on 40 pallet photos (lesson 3 of this module). Each box has a **score** from 0 to 1: how sure the detector is.

## 2. Intersection over union

IoU = the area where two boxes overlap, divided by the area that they cover together. 1 means the same box; 0 means no overlap.

> **Predict.** Box A is `[10, 10, 40, 40]` and box B is `[30, 20, 40, 40]`. Is their IoU above or below 0.5?

In [ ]:
def iou(a, b):
    ax1, ay1, bx1, by1 = a[0] + a[2], a[1] + a[3], b[0] + b[2], b[1] + b[3]
    w = max(0, min(ax1, bx1) - max(a[0], b[0]))
    h = max(0, min(ay1, by1) - max(a[1], b[1]))
    inter = w * h
    return inter / (a[2] * a[3] + b[2] * b[3] - inter)

A, B = [10, 10, 40, 40], [30, 20, 40, 40]
print("overlap", 20 * 30, "| together", 1600 + 1600 - 600, "| IoU", round(iou(A, B), 3), "| helper:", round(cp.iou(A, B), 3))

> **Check.** IoU is 0.231: below 0.5. A detector's box usually counts as right when its IoU with a true box is at least 0.5.

## 3. The boxes on one tight pallet

In [ ]:
def draw(sid, boxes, title):
    im = images[sid]
    fig, ax = plt.subplots(figsize=(7, 5.2))
    ax.imshow(Image.open("data/" + im["file_name"])); ax.axis("off")
    for x, y, w, h in truth[sid]:
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, color="lime", lw=1))
    for x, y, w, h in boxes:
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, color="red", lw=1, ls="--"))
    ax.set_title(f"{title}: {len(boxes)} red boxes, {im['count']} true (green)"); plt.show()

sid = [s for s, im in images.items() if im["crowding"] == "tight"][0]
mine = [p for p in preds if p["scene_id"] == sid]
draw(sid, [p["box"] for p in mine if p["score"] >= 0.6], "score at least 0.6")

## 4. Non-maximum suppression

A detector often gives two boxes for one object. NMS keeps the box with the best score and drops every other box whose IoU with it is above a line. A low line removes doubles, and it can also remove a real box that touches its neighbour.

> **Predict.** On the tight pallet, does an NMS line of 0.3 give more boxes or fewer than 0.7?

In [ ]:
kept = [p for p in mine if p["score"] >= 0.6]
for line in (0.3, 0.5, 0.7):
    keep = cp.nms([p["box"] for p in kept], [p["score"] for p in kept], line)
    print(f"NMS at IoU {line}: {len(keep)} boxes (true: {images[sid]['count']})")

## 5. Your turn: the count error on tight pallets

> **Your turn.** For the 10 **tight** test pallets, count the recorded boxes with a score of at least 0.6 (no extra NMS), and put the mean absolute difference from the true count in `tight_error` (boxes a photo, a number).

In [ ]:
errors = []
for s, im in images.items():
    if im["crowding"] == "tight":
        n = sum(1 for p in preds if p["scene_id"] == s and p["score"] >= 0.6)
        errors.append(abs(n - im["count"]))
tight_error = float(np.mean(errors))
print(round(tight_error, 2))

In [ ]:
expect('tight_error', tight_error, 6.9, tolerance=0.01)

## 6. Failure case: one box for two boxes

On a tight pallet two boxes touch. The detector sometimes draws one wide red box over both, and IoU says that red box is wrong for each of them. You might expect the count to be too low. It is not: at a score line of 0.5 the detector also keeps many extra boxes, so the count on tight pallets is too high, by 16.7 boxes on average. In the course's measurement, four wide kept boxes lay over touching boxes at a score line of 0.5: 38 pairs of touching boxes in all (on S-069, S-075 and S-084). The picture above uses 0.6, where there are none. To see one, draw S-069 with a score line of 0.5.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/vision/m04/boxes-and-iou/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Pretrained detectors](https://learning.nextia-ai.com/courses/vision/m04/pretrained-detectors/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/vision/m04/boxes-and-iou/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The photos are open photos (each with its author and licence in the manifests) or images made by our code; no photo shows a real person.